### Logistic Regression

### Linear Regression predicts a number. Logistic Regression predicts a probability and then makes a classification.

The problem

Imagine we're hiring candidates.

We have their interview score and whether they were selected:

Interview Score	Selected
20	      No (0)
30	      No (0)
40	      No (0)
50	      No (0)
60	      Yes (1)
70	      Yes (1)
80	      Yes (1)
90	      Yes (1)

Our goal:

If a new candidate gets 65, should we predict Selected or Not Selected?

This is a classification problem, so Logistic Regression is suitable.

First understand the idea

Logistic Regression doesn't directly say:

Selected = 1
Not Selected = 0

Instead, it gives a probability.

For example:

Score = 40 → Probability = 0.12
Score = 50 → Probability = 0.38
Score = 60 → Probability = 0.73
Score = 80 → Probability = 0.97

Then we use a threshold, commonly 0.5:

Probability >= 0.5 → 1 → Selected
Probability <  0.5 → 0 → Not Selected

So:

0.73 → Selected
0.38 → Not Selected

In [ ]:
from sklearn.linear_model import LogisticRegression

x = [[20],[30],[40],[50],[60],[70],[80],[90]]

y = [0,0,0,0,1,1,1,1]

model = LogisticRegression()

model.fit(x,y)

prediction = model.predict([[65]])

print(prediction [0])
 
probability = model.predict_proba([[65]]) # gives probabilities; meaning:  Not Selected = 25%  Selected = 75%

print(probability)

1
[[0.00349389 0.99650611]]


              Training Data
                   ↓
        ┌────────────────────┐
        │ Logistic Regression│
        └────────────────────┘
                   ↓
             New Score = 65
                   ↓
            Probability
                   ↓
              75% Selected
                   ↓
             Prediction = 1

Multiple features

Here's where Logistic Regression becomes much more interesting.

Instead of using only:

Interview Score

we can use:

Interview Score
CGPA
Projects
Coding Score

For example:

Interview	CGPA	Projects	Coding	Selected
50	6.5	1	40	0
60	7.0	2	50	0
70	7.5	3	65	1
80	8.0	4	75	1
90	9.0	5	90	1

In [6]:
from sklearn.linear_model import LogisticRegression

X = [
    [50, 6.5, 1, 40],
    [60, 7.0, 2, 50],
    [70, 7.5, 3, 65],
    [80, 8.0, 4, 75],
    [90, 9.0, 5, 90]
]

y = [0, 0, 1, 1, 1]

model = LogisticRegression()

model.fit(X, y)

# New candidate
candidate = [[75, 7.8, 3, 70]]

prediction = model.predict(candidate)
probability = model.predict_proba(candidate)

print("Prediction:", prediction[0])
print("Probability:", probability[0])

Prediction: 1
Probability: [0.00130772 0.99869228]


             Dataset
                |
       ┌────────┴────────┐
       ↓                 ↓
   Training Data      Test Data
       |                 |
       ↓                 ↓
   model.fit()      model.predict()

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = [
    [20], [30], [40], [50],
    [60], [70], [80], [90]
]

y = [0, 0, 0, 0, 1, 1, 1, 1]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42
)

model = LogisticRegression()

model.fit(X_train, y_train)

predictions = model.predict(X_test)

print("Actual:", y_test)
print("Predicted:", predictions)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Actual: [0, 1]
Predicted: [0 1]
Accuracy: 1.0


The complete beginner program

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#Data

X = [[20],[25],[30],[35],[40],[45],[50],[55],[60],[65],[70],[75],[80],[85],[90]]

y = [0,0,0,0,1,1,1,1,1,1,1,1,1,1,1]

# Split the data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.2,
    random_state=42
)

# Create a logistic regression model

model = LogisticRegression()

# Train the model on the training data

model.fit(X_train, y_train)

# Make predictions on the testing data

predictions = model.predict(X_test)

# Calculate the accuracy of the model

accuracy = accuracy_score(y_test, predictions)

print("Actual:", y_test)
print("Predicted:", predictions)
print("Accuracy:", accuracy)

#New candidate

new_score = [[68]]

prediction = model.predict(new_score)
probability = model.predict_proba(new_score)

print("Prediction:", prediction[0])
print("Probability:", probability[0])




Actual: [1, 1, 0]
Predicted: [1 1 0]
Accuracy: 1.0
Prediction: 1
Probability: [1.57917679e-10 1.00000000e+00]


In [10]:
"""
LOGISTIC REGRESSION — end-to-end on a realistic 'customer churn' dataset

GOAL: Predict the PROBABILITY that a customer will churn (cancel their
subscription), based on tenure, billing, support history, and contract type.
"""

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_auc_score, classification_report, confusion_matrix,
                              brier_score_loss, precision_recall_curve)
from sklearn.calibration import calibration_curve   # checks whether predicted probabilities are "truthful"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(7)   # reproducibility, same idea as before

# ----------------------------------------------------------------------
# 1. BUILD A REALISTIC SYNTHETIC CHURN DATASET
# ----------------------------------------------------------------------
n = 3000  # simulate 3000 customers

# tenure_months: how long the customer has been with the company.
# Exponential distribution: most customers are relatively new, fewer are
# very long-tenured (this mirrors real subscriber-base shapes).
tenure_months = np.random.exponential(scale=24, size=n).clip(1, 96)

# monthly_charges: their monthly bill. Normal distribution around $70.
monthly_charges = np.random.normal(70, 25, n).clip(15, 150)

# num_support_tickets: how many support complaints they filed.
# Poisson distribution is standard for "count" data (number of times
# something happened), e.g., number of tickets, number of late payments.
num_support_tickets = np.random.poisson(1.2, n)

# late_payments_12mo: how many times they paid late in the last year.
late_payments_12mo = np.random.poisson(0.8, n)

# contract_month_to_month: 1 if they're on a flexible no-lock-in contract,
# 0 if they're on a longer fixed-term contract. Binomial = coin-flip-like,
# here weighted so 40% of customers are month-to-month.
contract_month_to_month = np.random.binomial(1, 0.4, n)

# TRUE LOG-ODDS FORMULA — this is the hidden "ground truth" logistic
# regression relationship we're simulating (again, unknown in real life,
# defined here so we can check the model recovers it).
# NOTE: this is in LOG-ODDS space, not probability space — that's the
# defining feature of logistic regression (see the sigmoid step below).
log_odds = (
    -2.0                                # baseline log-odds when everything else is 0 (low churn baseline)
    - 0.05 * tenure_months              # longer tenure -> LOWER churn odds (negative coefficient)
    + 0.015 * monthly_charges           # higher bill -> HIGHER churn odds
    + 0.35 * num_support_tickets        # more complaints -> HIGHER churn odds
    + 0.55 * late_payments_12mo         # payment friction -> HIGHER churn odds
    + 1.1 * contract_month_to_month     # no lock-in contract -> MUCH easier to churn (biggest single effect)
)

# THE SIGMOID FUNCTION: converts log-odds (which can be any real number,
# -infinity to +infinity) into a clean probability between 0 and 1.
prob_churn = 1 / (1 + np.exp(-log_odds))

# Now simulate the actual yes/no churn OUTCOME using those probabilities.
# binomial(1, p) is like flipping a biased coin with probability p of "1" (churn).
# This adds realistic randomness — even a customer with 80% churn probability
# doesn't churn every single time; there's still a 20% chance they stay.
churn = np.random.binomial(1, prob_churn)

df = pd.DataFrame({
    "tenure_months": tenure_months,
    "monthly_charges": monthly_charges,
    "num_support_tickets": num_support_tickets,
    "late_payments_12mo": late_payments_12mo,
    "contract_month_to_month": contract_month_to_month,
    "churn": churn
})

print("="*70)
print("DATASET PREVIEW")
print("="*70)
print(df.head())
print(f"\nShape: {df.shape}")
print(f"\nChurn rate (class balance): {df['churn'].mean():.1%}")
# IMPORTANT: this is 36%, not 50% — real churn/fraud/default data is almost
# always imbalanced. This affects how we evaluate the model later.
print(df['churn'].value_counts())

# ----------------------------------------------------------------------
# 2. TRAIN/TEST SPLIT
# ----------------------------------------------------------------------
features = ["tenure_months", "monthly_charges", "num_support_tickets",
            "late_payments_12mo", "contract_month_to_month"]
X = df[features]
y = df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=7,
    stratify=y   # IMPORTANT: ensures both train and test sets keep the SAME
                 # 36%/64% churn ratio. Without this, a random split could
                 # accidentally put too many/too few churners in either set.
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Scaling here matters even more than in linear regression, because it makes
# the coefficients directly comparable (see odds ratio interpretation below).

# ----------------------------------------------------------------------
# 3. FIT LOGISTIC REGRESSION
# ----------------------------------------------------------------------
clf = LogisticRegression(
    C=1.0,                     # inverse of regularization strength (smaller C = stronger regularization)
    class_weight="balanced",   # CRITICAL for imbalanced data: automatically up-weights
                                # the minority class (churners) during training, so the
                                # model doesn't just learn to always predict "no churn"
    max_iter=1000               # number of optimization iterations allowed to converge
)
clf.fit(X_train_scaled, y_train)

print("\n" + "="*70)
print("MODEL COEFFICIENTS -> ODDS RATIOS (stakeholder-friendly interpretation)")
print("="*70)

# clf.coef_[0] gives the raw log-odds coefficients. We exponentiate them
# to get ODDS RATIOS, which are much easier to explain to a non-technical
# audience: "odds ratio of 1.7" = "73% higher odds of churn."
odds_ratios = np.exp(clf.coef_[0])

coef_table = pd.DataFrame({
    "feature": features,
    "coefficient (log-odds scale)": clf.coef_[0].round(3),
    "odds_ratio": odds_ratios.round(3)
}).sort_values("odds_ratio", ascending=False)
print(coef_table.to_string(index=False))
# odds_ratio > 1 -> increases churn odds
# odds_ratio < 1 -> decreases churn odds
# odds_ratio = 1 -> no effect

# ----------------------------------------------------------------------
# 4. EVALUATE
# ----------------------------------------------------------------------
# predict_proba gives probabilities for BOTH classes [P(no churn), P(churn)];
# we only want the second column — the probability of churning.
y_probs = clf.predict_proba(X_test_scaled)[:, 1]

# predict() applies the DEFAULT 0.5 threshold automatically:
# if P(churn) >= 0.5 -> predict "churn", else predict "no churn"
y_pred_default_thresh = clf.predict(X_test_scaled)

print("\n" + "="*70)
print("EVALUATION")
print("="*70)

# AUC-ROC: measures how well the model RANKS churners above non-churners,
# regardless of threshold. 0.5 = random guessing, 1.0 = perfect separation.
print(f"AUC-ROC        : {roc_auc_score(y_test, y_probs):.4f}")

# Brier score: average squared difference between predicted probability and
# actual outcome (0 or 1). Lower = better CALIBRATED probabilities
# (this is different from AUC — a model can rank well but still give
# badly-calibrated probability numbers).
print(f"Brier score    : {brier_score_loss(y_test, y_probs):.4f}  (lower = better, 0=perfect)")

print("\nClassification report (default 0.5 threshold):")
# precision: of everyone we PREDICTED would churn, what % actually did?
# recall: of everyone who ACTUALLY churned, what % did we catch?
# f1-score: harmonic mean of precision and recall (balances both)
print(classification_report(y_test, y_pred_default_thresh, target_names=["No Churn", "Churn"]))

cm = confusion_matrix(y_test, y_pred_default_thresh)
print("Confusion matrix:")
print(f"                 Predicted No   Predicted Yes")
print(f"Actual No        {cm[0][0]:>10}    {cm[0][1]:>10}")   # top-left: correct "no churn" calls; top-right: false alarms
print(f"Actual Yes       {cm[1][0]:>10}    {cm[1][1]:>10}")   # bottom-left: MISSED churners; bottom-right: correctly caught churners

# ----------------------------------------------------------------------
# 5. THRESHOLD TUNING BASED ON BUSINESS COST
# ----------------------------------------------------------------------
# The 0.5 threshold is a MATHEMATICAL default, not a business-optimal one.
# precision_recall_curve computes precision & recall at EVERY possible
# threshold, so we can pick the one that best matches real-world costs.
precisions, recalls, thresholds = precision_recall_curve(y_test, y_probs)

# F2 score = like F1, but weights RECALL more heavily than precision.
# We use this here because in churn: missing a real churner (false negative)
# is usually more costly than wasting a retention offer on someone who
# wouldn't have churned anyway (false positive).
f2_scores = (5 * precisions * recalls) / (4 * precisions + recalls + 1e-9)
# (the "+1e-9" avoids a divide-by-zero error when both precision and recall are 0)

best_idx = np.argmax(f2_scores[:-1])   # find the threshold that maximizes F2
# (we drop the last element because precision_recall_curve returns one extra
# point that doesn't correspond to a real threshold)
best_threshold = thresholds[best_idx]

print(f"\nBusiness-cost-optimized threshold (F2, favors catching churners): {best_threshold:.3f}")
print(f"  At this threshold -> Precision: {precisions[best_idx]:.3f}, Recall: {recalls[best_idx]:.3f}")

# ----------------------------------------------------------------------
# 6. CALIBRATION CHECK
# ----------------------------------------------------------------------
# Groups predictions into bins (e.g., "all predictions around 0.7") and
# compares: "of everyone I gave a 0.7 probability to, did ~70% actually churn?"
# This is what "calibration" means, and it's DIFFERENT from accuracy/AUC.
prob_true, prob_pred = calibration_curve(y_test, y_probs, n_bins=10, strategy="quantile")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# PLOT 1: Calibration curve
axes[0].plot(prob_pred, prob_true, marker='o', label="Model")
axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray', label="Perfect calibration")
# The gray dashed diagonal is the IDEAL — if our model's line matches it,
# predicted probabilities can be trusted at face value.
axes[0].set_xlabel("Mean predicted probability")
axes[0].set_ylabel("Observed churn rate")
axes[0].set_title("Calibration Curve")
axes[0].legend()

# PLOT 2: Precision & recall as a function of threshold
axes[1].plot(thresholds, precisions[:-1], label="Precision")
axes[1].plot(thresholds, recalls[:-1], label="Recall")
axes[1].axvline(best_threshold, color='red', linestyle='--', label=f"Chosen threshold={best_threshold:.2f}")
axes[1].set_xlabel("Decision threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("Precision/Recall vs Threshold")
axes[1].legend()

plt.tight_layout()
plt.savefig("logistic_regression_diagnostics.png", dpi=130)
print("\nSaved diagnostic plot -> logistic_regression_diagnostics.png")

DATASET PREVIEW
   tenure_months  monthly_charges  num_support_tickets  late_payments_12mo  \
0       1.905046        97.461118                    1                   1   
1      36.330208       100.169345                    1                   0   
2      13.847565        62.676530                    1                   0   
3      30.850045        88.350159                    0                   1   
4      91.589669        33.618101                    0                   0   

   contract_month_to_month  churn  
0                        0      0  
1                        0      0  
2                        1      0  
3                        0      0  
4                        0      0  

Shape: (3000, 6)

Churn rate (class balance): 36.1%
churn
0    1916
1    1084
Name: count, dtype: int64

MODEL COEFFICIENTS -> ODDS RATIOS (stakeholder-friendly interpretation)
                feature  coefficient (log-odds scale)  odds_ratio
contract_month_to_month                         0.548  